Webscraper for open PostDoc positions at some German universities

In [1]:
# Import libraries
import requests
import time
from bs4 import BeautifulSoup
from urllib.parse import urljoin


In [2]:
# Define headers, keywords, and sleep time
headers = {"User-Agent": "Mozilla/5.0 (JobScraper; contact:me@myprovider.com)"}
in_keywords = ["psycho", "neuro"]
out_keywords = ["promotion", "doktorand", "predoc", "arzt", "therapeut", "elektroniker"]
sleeptime = 1

In [3]:
# Define IPU scraper
def scrape_ipu(url, headers, sleeptime, in_keywords=None, out_keywords=None):

    base_url = "https://www.ipu-berlin.de"
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "lxml")

    jobs = []

    filtered_jobs = soup.select("ul.bloglist.bloglist_3 li") # IPU-specific structure

    for li in filtered_jobs:
        # Text from <h2> + <div class="infos">
        title = li.find("h2").get_text(strip=True)
        infos_short = li.find("div", class_="infos").get_text(strip=True) if li.find("div", class_="infos") else ""
        combined_text = f"{title} {infos_short}".lower()

        # Check keywords
        ipu_kws = ["wissenschaft", "post"] # needs some extra keywords to avoid IPU-specific irrelevant jobs
        if not any(kw.lower() in combined_text for kw in ipu_kws):
                continue

        if any(k.lower() in title.lower() for k in out_keywords):
            continue

        # Date
        dates = [d.get_text(strip=True) for d in li.find_all("div", class_="event-date") if d.get_text(strip=True)]

        # Link
        link = urljoin(base_url, li.find("a")["href"])

        # Job details
        resp = requests.get(link, headers=headers)
        if resp.status_code == 200:
            detail_soup = BeautifulSoup(resp.text, "lxml")
            meta_desc = detail_soup.find("meta", attrs={"name": "description"})
            infos = meta_desc["content"] if meta_desc else ""
        else:
            infos = ""

        jobs.append({
            "title": title,
            "dates": dates,
            "link": link,
            "infos": " ".join(infos.split())  # clean line breaks
        })

        time.sleep(sleeptime)

    return jobs


In [4]:
# Define HTW scraper
def scrape_htw(url, headers, sleeptime, in_keywords=None, out_keywords=None):

    base_url = "https://www.htw-berlin.de"
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "lxml")

    jobs = []

    filtered_jobs = soup.select("dt")

    for dt in filtered_jobs:

        a_tag = dt.find("a")
        if not a_tag:
            continue
        title = a_tag.get_text(strip=True)
        link = a_tag["href"]

        # Info
        dd = dt.find_next_sibling("dd")
        infos = dd.get_text(strip=True) if dd else ""

        # Keywords filter
        # combined_text = f"{title} {infos}".lower()
        # if not any(kw.lower() in combined_text for kw in in_keywords):
        #   continue

        # Put in dictionary
        jobs.append({
            "title": title,
            "infos": infos,
            "link": link,
        })

        time.sleep(sleeptime)

    return jobs

In [5]:
# Define HWR scraper
def scrape_hwr(url, headers, sleeptime, in_keywords=None, out_keywords=None):
    base_url = "https://www.hwr-berlin.de"
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "lxml")

    jobs = []

    section = soup.select_one("#c9832")
    title   = section.get_text(" ", strip=True)
    content = section.find_next_sibling("div")
    infos   = content.get_text(" ", strip=True)

    # Put in dictionary
    jobs.append({
        "title": title,
        "infos": infos,
        })

    time.sleep(sleeptime)

    return jobs

In [6]:
# Define Uni Greifswald Scraper
def scrape_ugr(url, headers, sleeptime, in_keywords=None, out_keywords=None):
  base_url = "https://www.uni-greifswald.de"
  response = requests.get(url, headers=headers)
  soup = BeautifulSoup(response.text, "lxml")

  jobs = []

  filtered_jobs = soup.select("div.plugin-list__content--full-width")

  for item in filtered_jobs:
      a_tag = item.select_one("a.teaserbox")
      if not a_tag:
          continue

      title_tag = a_tag.select_one("h3.hyphenate.h2-style")
      if not title_tag:
          continue

      title = title_tag.get_text(strip=True)

      # Deadline
      deadline = ""
      p = item.select_one("div.teaserbox__content p")
      if p:
        times = p.find_all("time")
        if len(times) >= 2:
          deadline = times[1].get_text(strip=True)

      link = base_url + a_tag["href"]

      # keyword filters
      if any(k.lower() in title.lower() for k in out_keywords):
            continue

      jobs.append({
        "title": title,
        "deadline": deadline,
        "link": link
          })

      time.sleep(sleeptime)
  return jobs


In [7]:
# Define Uni Jena Scraper
def scrape_uje(url, headers, sleeptime, in_keywords=None, out_keywords=None):
  base_url = "https://www.uni-jena.de"
  response = requests.get(url, headers=headers)
  soup = BeautifulSoup(response.text, "lxml")

  jobs = []
  filtered_jobs = soup.select("ol.entries li.entry.fsv")

  for entry in filtered_jobs:
      a_tag = entry.select_one("div.content a[href]")
      if not a_tag:
          continue

      title = a_tag.get_text(strip=True)
      link = a_tag["href"]
      faculty_tag = entry.select_one("div.content div.institution")
      faculty = faculty_tag.get_text(strip=True) if faculty_tag else ""
      title = f"{faculty}\n{title}" if faculty else title

      date_tag = entry.select_one("div.content div.date-number time")
      deadline = date_tag.get_text(strip=True) if date_tag else ""

      # Keyword filters
      if not any(kw in title.lower() for kw in in_keywords):
          continue
      if any(k.lower() in title.lower() for k in out_keywords):
          continue

      # Add to dict
      jobs.append({
          "title": title,
          "deadline": deadline,
          "link": link
      })
      time.sleep(sleeptime)
  return jobs

In [8]:
# Define Uni Leipzig scraper
def scrape_ulip(url, headers, sleeptime, in_keywords=None, out_keywords=None):
  base_url = "https://www.uni-leipzig.de"
  response = requests.get(url, headers=headers)
  soup = BeautifulSoup(response.text, "lxml")

  filtered_jobs = soup.select("article.searchresult")
  jobs = []
  for article in filtered_jobs:
    a_tag = article.select_one("a.blocklink")

    # Link
    link = a_tag["href"]

    # Title
    title_tag = article.select_one("h3.h2-teaser-lowercase")
    title = title_tag.get_text(strip=True) if title_tag else ""

    # Faculty
    faculty_tag = article.select_one("p.subheadline")
    faculty = faculty_tag.get_text(strip=True) if faculty_tag else ""

    # Faculty + title
    title = f"{faculty}\n{title}" if faculty else title

    # keywords filter
    if not any(kw in title.lower() for kw in in_keywords):
      continue

    # Add to dict
    jobs.append({
    "title": title,
    "link": link
      })

    time.sleep(sleeptime)
  return jobs

In [9]:
# Define TU Dresden scraper
def scrape_tud(url, headers, sleeptime, in_keywords=None, out_keywords=None):
  base_url = "https://tu-dresden.de"
  response = requests.get(url, headers=headers)
  soup = BeautifulSoup(response.text, "lxml")

  filtered_jobs = soup.select("div.row.collapse h2")
  jobs = []

  relevant_sections = ["Exzellenzcluster", "Fakultät Psychologie"]

  jobs = []

  for section_title in relevant_sections:
      # look for h2
      h2 = soup.find("h2", string=section_title)
      if not h2:
          continue  # continue if no open jobs in this section

      # associated ul
      ul = h2.find_next_sibling("ul")
      if not ul:
          continue

      # all li in ul
      for li in ul.find_all("li"):
          # Link
          a_tag = li.find("a")
          link = a_tag["href"] if a_tag else ""

          # Title
          title = a_tag.get_text(strip=True) if a_tag else ""

          # Faculty to title
          full_title = f"{section_title}\n{title}"

          # Deadline
          br_texts = li.get_text(separator="\n").split("\n")
          deadline = ""
          for line in br_texts:
              if "Bewerbungsschluss" in line:
                  deadline = f"{line.replace('Bewerbungsschluss:', '').strip()}"

          # Keywords
          if not any(kw in full_title.lower() for kw in in_keywords):
              continue

          jobs.append({
              "title": full_title,
              "deadline": deadline,
              "link": link
          })

          time.sleep(sleeptime)
  return jobs

In [10]:
# Define OVGU Magdeburg scraper
def scrape_ovgu(url, headers, sleeptime, in_keywords=None, out_keywords=None):
  base_url = "https://www.ovgu.de"
  response = requests.get(url, headers=headers)
  soup = BeautifulSoup(response.text, "lxml")

  filtered_jobs = soup.select("a.job-entry")
  jobs = []

  for job in filtered_jobs:

    link = job["href"]

    title_tag = job.find("h2")
    title = title_tag.get_text(separator="\n").strip() if title_tag else ""

    faculty_tag = job.find("p", class_="job-ou")
    faculty = faculty_tag.get_text(separator="\n").strip() if faculty_tag else ""
    title = f"{faculty}\n{title}" if faculty else title

    deadline = ""
    footer = job.find("div", class_="job-footer")
    if footer:
      for p in footer.find_all("p"):
        if "Bewerbung" in p.get_text():
          deadline = f"{p.get_text(strip=True).replace('Bewerbung bis', '').strip()}"

    # Keywords-Filter
    if not any(kw in title.lower() for kw in in_keywords):
      continue
    if any(k.lower() in title.lower() for k in out_keywords):
            continue

    jobs.append({
          "title": title,
          "deadline": deadline,
          "link": link
      })
    time.sleep(sleeptime)

  return jobs


In [11]:
# Define FU Berlin scraper
def scrape_fub(url, headers, sleeptime, in_keywords=None, out_keywords=None):
  base_url = "https://www.fu-berlin.de"
  response = requests.get(url, headers=headers)
  soup = BeautifulSoup(response.text, "lxml")

  filtered_jobs = soup.select_one("div.box.box-job-offers-list")

  jobs = []

  title = filtered_jobs.find("h1").get_text(strip=True)
  info = filtered_jobs.find("em").get_text(strip=True)
  jobs.append({
      "title": title,
      "infos": info
  })

  time.sleep(sleeptime)

  return jobs


In [12]:
# HU Berlin scraper
def scrape_hub(url, headers, sleeptime, in_keywords=None, out_keywords=None):
    base_url = "https://www.hu-berlin.de"
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "lxml")

    job_divs = soup.select("div.job-list-item")
    jobs = []

    for job_div in job_divs:
        a_tag = job_div.find("a")
        if not a_tag:
            continue

        # Link
        link = base_url + a_tag["href"]

        # Title
        title_tag = a_tag.select_one("h3.headline-scala")
        title = title_tag.get_text(strip=True) if title_tag else ""

        # Info
        subtitle_tag = a_tag.select_one("p.list-item-subtitle strong")
        subtitle = subtitle_tag.get_text(strip=True) if subtitle_tag else ""

        # Faculty
        faculty_divs = a_tag.select("div.flex-grow-1 > div > div.mb-1")
        faculty = "\n".join(div.get_text(strip=True) for div in faculty_divs)

        # Combine
        lines = []
        if faculty:
            lines.append(faculty)
        if title:
            lines.append(title)
        if subtitle:
            lines.append(subtitle)

        title = "\n".join(lines)

        # Deadline
        deadline_tag = a_tag.select_one("strong.deadline-text")
        deadline = ""
        if deadline_tag and deadline_tag.next_sibling:
            deadline = deadline_tag.next_sibling.strip()

        # Keywords filter
        if in_keywords:
            if not any(kw in title.lower() for kw in in_keywords):
                continue

        jobs.append({
            "title": title,
            "deadline": deadline,
            "link": link
        })
        time.sleep(sleeptime)

    return jobs


In [13]:
# Define TU Berlin scraper
def scrape_tub(url, headers, sleeptime, in_keywords=None, out_keywords=None):
  base_url = "https://www.jobs.tu.berlin"
  response = requests.get(url, headers=headers)
  soup = BeautifulSoup(response.text, "lxml")

  filtered_jobs = soup.select("div.border-b.py-4")

  jobs = []

  def clean_text(text):
      return text.replace("\xad", "").replace("\u00ad", "").strip()

  for box in soup.select("div.border-b.py-4"):
      # Titel + Link
      a = box.select_one("div.text-lg a")
      title = a.get_text(strip=True) if a else ""
      title = clean_text(a.get_text())
      link = base_url + a["href"] if a and a.has_attr("href") else ""

      # Fakultät / Institut
      org = box.select_one("div.mt-2")
      org_text = org.get_text(strip=True) if org else ""
      title = f"{org_text}\n{title}" if org_text else title

      # Deadline
      deadline = ""
      for li in box.select("ul li"):
          label = li.select_one("span")
          if label and "Bewerbungsfrist" in label.get_text():
              deadline = li.get_text(strip=True).replace("Bewerbungsfrist:", "").strip()

       # Keywords filter
      if in_keywords:
        if not any(kw in title.lower() for kw in in_keywords):
          continue

      jobs.append({
          "title": title,
          "deadline": deadline,
          "url": link
      })
      time.sleep(sleeptime)
  return jobs


In [14]:
# Define MPI Leipzig scraper
def scrape_mpil(url, headers, sleeptime, in_keywords=None, out_keywords=None):
  base_url = "https://www.cbs.mpg.de"
  response = requests.get(url, headers=headers)
  soup = BeautifulSoup(response.text, "lxml")

  iframe = soup.find("iframe")
  iframe_url = iframe["src"]
  response = requests.get(iframe_url, headers=headers)
  iframe_soup = BeautifulSoup(response.text, "lxml")
  jobs = []

  for tr in iframe_soup.select("tr[class^='table-as-list__contentrow']"):
      # Job-Link und Titel
      a = tr.select_one("h3 a.HSTableLinkSubTitle")
      if not a:
          continue
      title = a.get_text(strip=True)
      link = "https://recruitingapp-5218.de.umantis.com" + a["href"]

      # Faculty
      institute_li = tr.find("li", attrs={"aria-label": True, "class": "form_content_paragraph"})
      institute = institute_li.get_text(strip=True) if institute_li else "Max-Planck-Institut"

      if out_keywords and any(k.lower() in title.lower() for k in out_keywords):
            continue

      jobs.append({
          "title": f"{institute}\n{title}",
          "link": link
      })

      time.sleep(sleeptime)

  return jobs


In [15]:
# Define Uni Potsdam scraper
def scrape_upot(url, headers, sleeptime, in_keywords=None, out_keywords=None):
  base_url = "https://www.uni-potsdam.de"
  response = requests.get(url, headers=headers)
  soup = BeautifulSoup(response.text, "lxml")

  jobs = []

  for a in soup.select('a[href*="2_akadPersonal"]'):
    title = a.get_text(strip=True)
    if not title:
      continue

    parts = title.split(",", 1)
    if len(parts) == 2:
      second_line = parts[0].strip()
      first_line = parts[1].strip()
      title = f"{first_line}\n{second_line}"

    link = base_url + a["href"]

    # Keywords filter
    if any(k in title.lower() for k in out_keywords):
      continue
    if not any(k.lower() in title.lower() for k in in_keywords):
      continue

    jobs.append({
      "title": title,
      "link": link
    })

    time.sleep(sleeptime)
  return jobs


In [16]:
# Define websites to scrape

websites = [
    {
        "name": "FU Berlin",
        "url": "https://www.fu-berlin.de/universitaet/beruf-karriere/jobs/wiss/12_fb-erziehungswissenschaft-und-psychologie/index.html",
        "scraper": scrape_fub
    },
    {
        "name": "HU Berlin",
        "url": "https://www.hu-berlin.de/universitaet/arbeiten-an-der-hu/stellenangebote?tx_jobapplications%5Baction%5D=search&tx_jobapplications%5Bcontroller%5D=Posting&tx_jobapplications%5Bconstraint%5D%5BjobCategories%5D%5B%5D=6&tx_jobapplications%5Bconstraint%5D%5Borganizations%5D%5B%5D=893221&tx_jobapplications%5Bconstraint%5D%5BenableInternalOffers%5D=&tx_jobapplications%5Bt%5D=1766227099577",
        "scraper": scrape_hub
    },
    {
        "name": "TU Berlin",
        "url": "https://www.jobs.tu-berlin.de/stellenausschreibungen?filter%5Bfulltextsearch%5D=&filter%5Bworktype_tub%5D%5B%5D=wiss-dauer&filter%5Bworktype_tub%5D%5B%5D=wiss-mlehr&filter%5Bworktype_tub%5D%5B%5D=wiss-olehr&filter%5Bworktype_tub%5D%5B%5D=wiss-pdoc",
        "scraper": scrape_tub
    },
    {
        "name": "IPU Berlin",
        "url": "https://www.ipu-berlin.de/stellenangebote",
        "scraper": scrape_ipu
    },
    {
        "name": "HTW Berlin",
        "url": "https://www.htw-berlin.de/karriere/jobs-an-der-htw-berlin/wissenschaftliches-personal",
        "scraper": scrape_htw
    },
    {
        "name": "HWR Berlin",
        "url": "https://www.hwr-berlin.de/hwr-berlin/stellenangebote",
        "scraper": scrape_hwr
    },
    {
        "name": "Uni Greifswald",
        "url": "https://www.uni-greifswald.de/universitaet/information/stellenausschreibungen/wissenschaftliches-personal",
        "scraper": scrape_ugr
    },
    {
        "name": "Uni Potsdam", # limited positions
        "url": "https://www.uni-potsdam.de/de/verwaltung/dezernat3/stellenausschreibungen/befristete-stellen-fuer-akademisches-personal",
        "scraper": scrape_upot
    },
    {
        "name": "Uni Potsdam", # unlimited positions
        "url": "https://www.uni-potsdam.de/de/verwaltung/dezernat3/stellenausschreibungen/unbefristete-stellen-fuer-akademisches-personal",
        "scraper": scrape_upot
    },
    {
        "name": "Uni Jena",
        "url": "https://www.uni-jena.de/122166/stellenmarkt?category=3",
        "scraper": scrape_uje
    },
    {
        "name": "Uni Leipzig",
        "url": "https://www.uni-leipzig.de/universitaet/arbeiten-an-der-universitaet-leipzig/stellenausschreibungen?tx_news_pi1[__referrer][@extension]=News&tx_news_pi1[__referrer][@controller]=News&tx_news_pi1[__referrer][@action]=searchForm&tx_news_pi1[__referrer][arguments]=YTo0OntzOjY6ImFjdGlvbiI7czo0OiJsaXN0IjtzOjEwOiJjb250cm9sbGVyIjtzOjQ6Ik5ld3MiO3M6MTE6ImN1cnJlbnRQYWdlIjtzOjE6IjMiO3M6ODoicGx1Z2luSWQiO3M6NjoiOTA2MDI2Ijt9778c067f4943c1170647c0ac572d8bec0181e7a4&tx_news_pi1[__referrer][@request]=%7B%22%40extension%22%3A%22News%22%2C%22%40controller%22%3A%22News%22%2C%22%40action%22%3A%22searchForm%22%7Dd6b10075d920cf044871a82faa3c4e93fcfe73c2&tx_news_pi1[__trustedProperties]=%7B%22search%22%3A%7B%22subject%22%3A1%2C%22establishments%22%3A%5B1%2C1%2C1%2C1%5D%2C%22categories%22%3A%5B1%2C1%2C1%5D%7D%2C%22newsSearchSubmitted%22%3A1%7D6e8ba3e2aec15186a24bd0b71b804720452b8259&tx_news_pi1[search][subject]=&tx_news_pi1[search][establishments]=&tx_news_pi1[search][establishments][]=2&tx_news_pi1[search][categories]=&tx_news_pi1[search][categories][]=722&tx_news_pi1[newsSearchSubmitted]=1",
        "scraper": scrape_ulip
    },
    {
        "name": "MPI Leipzig",
        "url": "https://www.cbs.mpg.de/stellenmarkt/stellenangebote",
        "scraper": scrape_mpil
    },
    {
        "name": "TU Dresden",
        "url": "https://tu-dresden.de/tu-dresden/arbeiten-an-der-tud/stellenangebote/stellenangebote?kat=2&style=cms2",
        "scraper": scrape_tud
    },
    {
        "name": "OVGU Magdeburg",
        "url": "https://www.ovgu.de/Karriere_wissenschaftlichesPersonal.html",
        "scraper": scrape_ovgu
    }
]




In [17]:
# Scrape & collect jobs
all_jobs = []

for site in websites:
    print(f"Scraping {site['name']} ...")
    jobs = site["scraper"](site["url"], headers=headers, sleeptime=sleeptime, in_keywords=in_keywords, out_keywords=out_keywords)

    # Include university name
    for job in jobs:
        job['university'] = site['name']

    all_jobs.extend(jobs)

# print(f"Total number of jobs: {len(all_jobs)}")

Scraping FU Berlin ...
Scraping HU Berlin ...
Scraping TU Berlin ...
Scraping IPU Berlin ...
Scraping HTW Berlin ...
Scraping HWR Berlin ...
Scraping Uni Greifswald ...
Scraping Uni Potsdam ...
Scraping Uni Potsdam ...
Scraping Uni Jena ...
Scraping Uni Leipzig ...
Scraping MPI Leipzig ...
Scraping TU Dresden ...
Scraping OVGU Magdeburg ...


In [18]:
# Print job list
for job in all_jobs:
    print(f"{job.get('university', '---')}")
    print(f"Titel: {job.get('title', '---')}")

    if job.get('deadline'):
        print(f"Bewerbungsfrist: {job['deadline']}")

    if job.get('infos'):
        print(f"Infos: {job['infos']}")

    print(f"Link: {job.get('link', '---')}")
    print()


FU Berlin
Titel: Ausschreibungen für wissenschaftliches Personal am Fachbereich Erziehungswissenschaft und Psychologie
Infos: Zur Zeit sind keine aktuellen Stellenausschreibungen verfügbar
Link: ---

HU Berlin
Titel: Lebenswissenschaftliche Fakultät
Wissenschaftliche*r Mitarbeiter*in (m/w/d) im Bereich Klinische Psychologie
1/2-Teilzeitbeschäftigung - E 13 TV-L HU (Drittmittelfinanzierung befristet bis 30.06.2030)
Bewerbungsfrist: 04.02.2026
Link: https://www.hu-berlin.de/universitaet/arbeiten-an-der-hu/stellenangebote/details/wissenschaftlicher-mitarbeiterin-m-w-d-im-bereich-klinische-psychologie-dr-010-26

HU Berlin
Titel: Lebenswissenschaftliche Fakultät
Wissenschaftliche*r Mitarbeiter*in (m/w/d) im Bereich Klinische Psychologie
vorauss. 75 v. H. d. regelm. Arbeitszeit - E 13 TV-L HU (Drittmittelfinanzierung befristet bis 31.12.2031)
Bewerbungsfrist: 04.02.2026
Link: https://www.hu-berlin.de/universitaet/arbeiten-an-der-hu/stellenangebote/details/wissenschaftlicher-mitarbeiterin-m-w